# Model Complexity Comparison

**Research question:** Does increasing model complexity improve rental prediction performance?

This experiment compares a mean baseline, linear models, and tree ensembles with the same five baseline features and fixed temporal split. The goal is to examine linear versus nonlinear model behavior, not only select the lowest RMSE. A tree model's gain may suggest that the linear form is insufficient, but does not identify the cause.

Run Jupyter from the repository root. Dependencies are listed in `requirements.txt`.

In [40]:
import sys
import time
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').is_dir():
    raise RuntimeError('Start Jupyter from the repository root.')
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import Pipeline

from src.evaluation import evaluate
from src.features import BASELINE_FEATURES, make_features, make_preprocessor
from src.validation import load_data

## Shared data and features

`load_data()` reads the original training file, applies shared cleaning, and returns the fixed January 2021–April 2024 training period and May 2024–March 2025 validation period. No test rows are used. Every model receives exactly the five `BASELINE_FEATURES`; imputation, scaling, and one-hot encoding are fitted within its Pipeline. Dense preprocessor output is used because CatBoost requires dense input here.

In [41]:
train_df, valid_df = load_data()
X_train = make_features(train_df)
X_valid = make_features(valid_df)
y_train = train_df['MONTHLY_RENT']
y_valid = valid_df['MONTHLY_RENT']

assert tuple(X_train.columns) == BASELINE_FEATURES
assert tuple(X_valid.columns) == BASELINE_FEATURES
assert X_train.index.equals(y_train.index)
assert X_valid.index.equals(y_valid.index)
assert X_train.columns.equals(X_valid.columns)

print(f"Training:   {len(train_df):,} rows, {train_df['RENT_APPROVAL_DATE'].min()} to {train_df['RENT_APPROVAL_DATE'].max()}")
print(f"Validation: {len(valid_df):,} rows, {valid_df['RENT_APPROVAL_DATE'].min()} to {valid_df['RENT_APPROVAL_DATE'].max()}")
print('Features:', list(X_train.columns))

Training:   117,988 rows, 2021-01 to 2024-04
Validation: 32,012 rows, 2024-05 to 2025-03
Features: ['floor_area_sqm', 'flat_age', 'month_index', 'TOWN', 'FLAT_TYPE']


## Model set and initial parameters

Tree models use moderate starting configurations and fixed random seeds. Training time measures Pipeline `fit()`, including preprocessing, and excludes prediction. These are starting points for comparison, not an exhaustive tuning study. Extra Trees and histogram gradient boosting add different tree-building strategies. The native CatBoost run uses the same five columns but handles `TOWN` and `FLAT_TYPE` as categorical inputs instead of one-hot encoding them. Run all cells to refresh the table; any saved output from an earlier run does not include the newly added models.

In [42]:
models = {
    'Dummy Regressor': DummyRegressor(strategy='mean'),
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(
        n_estimators=200, max_depth=18, min_samples_leaf=5,
        random_state=42, n_jobs=-1
    ),
    'Extra Trees': ExtraTreesRegressor(
        n_estimators=300, max_depth=18, min_samples_leaf=5,
        random_state=42, n_jobs=-1
    ),
    'HistGradientBoosting': HistGradientBoostingRegressor(
        max_iter=300, learning_rate=0.08, max_leaf_nodes=31,
        l2_regularization=1.0, random_state=42
    ),
    'LightGBM': LGBMRegressor(
        n_estimators=500, learning_rate=0.05, num_leaves=31,
        random_state=42, n_jobs=-1, verbosity=-1
    ),
    'CatBoost': CatBoostRegressor(
        iterations=500, learning_rate=0.05, depth=8,
        loss_function='RMSE', random_seed=42, thread_count=-1, verbose=False
    ),
}
model_family = {
    'Dummy Regressor': 'Constant baseline',
    'Linear Regression': 'Linear',
    'Ridge Regression': 'Regularized linear',
    'Random Forest': 'Tree ensemble',
    'Extra Trees': 'Randomized tree ensemble',
    'HistGradientBoosting': 'Histogram gradient boosting',
    'LightGBM': 'Gradient-boosted trees',
    'CatBoost': 'Gradient-boosted trees',
}

In [43]:
results = []
fitted_models = {}
for name, regressor in models.items():
    if name == 'CatBoost (native categorical)':
        # Same five input columns; CatBoost handles the two categoricals directly.
        model_X_train = X_train.copy()
        model_X_valid = X_valid.copy()
        for column in ('TOWN', 'FLAT_TYPE'):
            model_X_train[column] = model_X_train[column].fillna('MISSING').astype(str)
            model_X_valid[column] = model_X_valid[column].fillna('MISSING').astype(str)
        experiment = regressor
    else:
        # Fresh, identical preprocessing fitted separately on training data.
        preprocessor = make_preprocessor().set_params(sparse_threshold=0)
        experiment = Pipeline([('preprocess', preprocessor), ('model', regressor)])
        model_X_train = X_train
        model_X_valid = X_valid
    start = time.perf_counter()
    experiment.fit(model_X_train, y_train)
    fit_seconds = time.perf_counter() - start
    predictions = experiment.predict(model_X_valid)
    metrics = evaluate(y_valid, predictions)
    assert np.isfinite(predictions).all(), f'{name} produced non-finite predictions'
    assert all(np.isfinite(value) for value in metrics.values()), f'{name} produced non-finite metrics'
    results.append({
        'Model': name, 'Family': model_family[name], **metrics,
        'Training Time (s)': fit_seconds,
    })
    fitted_models[name] = experiment
    print(f"{name}: fit {fit_seconds:.2f}s | RMSE {metrics['RMSE']:.2f}")

results_df = pd.DataFrame(results).sort_values('RMSE').reset_index(drop=True)
results_df.insert(0, 'RMSE Rank', np.arange(1, len(results_df) + 1))

Dummy Regressor: fit 0.15s | RMSE 839.61
Linear Regression: fit 0.20s | RMSE 616.28
Ridge Regression: fit 0.21s | RMSE 616.29
Random Forest: fit 7.53s | RMSE 493.00
Extra Trees: fit 16.50s | RMSE 492.94
HistGradientBoosting: fit 1.14s | RMSE 484.21
LightGBM: fit 0.85s | RMSE 483.09
CatBoost: fit 4.13s | RMSE 482.68


## Performance comparison and interpretation

RMSE and MAE use rental currency units; positive Mean Error indicates over-prediction. Compare the linear rows with tree models to assess whether the same baseline features benefit from nonlinear splits or interactions. A performance gap alone does not establish which feature relationship explains it.

In [44]:
# Export a reusable performance table beside this notebook.
output_dir = PROJECT_ROOT / 'experiments'
csv_path = output_dir / 'model_comparison_results.csv'
markdown_path = output_dir / 'model_comparison_results.md'
tabm_results_path = output_dir / 'tabm_hdb_rent_results.csv'
results_df = results_df.drop(columns=['RMSE Rank'])
if tabm_results_path.exists():
    tabm_results = pd.read_csv(tabm_results_path)
    results_df = pd.concat([results_df, tabm_results], ignore_index=True)
results_df = results_df.sort_values('RMSE').reset_index(drop=True)
results_df.insert(0, 'RMSE Rank', np.arange(1, len(results_df) + 1))
display(results_df.round({
    'RMSE': 2, 'MAE': 2, 'R2': 4, 'Mean Error': 2, 'Training Time (s)': 2,
}))
results_df.to_csv(csv_path, index=False)
table = results_df.round({
    'RMSE': 2, 'MAE': 2, 'R2': 4, 'Mean Error': 2, 'Training Time (s)': 2,
})
headers = list(table.columns)
markdown_rows = ['| ' + ' | '.join(headers) + ' |', '| ' + ' | '.join(['---'] * len(headers)) + ' |']
markdown_rows += [
    '| ' + ' | '.join(str(value) for value in row) + ' |'
    for row in table.itertuples(index=False, name=None)
]
note = (
    '\n\nNote: TabM uses the same fixed train/validation dates but a larger, different feature set than the five-feature baseline models. '
    'Its run time is approximate (about 155 seconds, inferred from the progress log). Mean Error was not included in the supplied TabM output.'
)
markdown_path.write_text(
    '# Model Comparison Results\n\n' + '\n'.join(markdown_rows) + note + '\n',
    encoding='utf-8',
)
print(f'Saved {csv_path.relative_to(PROJECT_ROOT)} and {markdown_path.relative_to(PROJECT_ROOT)}')

,RMSE Rank,Model,Family,RMSE,MAE,R2,Mean Error,Training Time (s)
0,1,CatBoost,Gradient-boosted trees,482.68,358.50,0.4602,-43.92,4.13
1,2,LightGBM,Gradient-boosted trees,483.09,359.34,0.4593,-44.02,0.85
2,3,HistGradientBoosting,Histogram gradient boosting,484.21,360.63,0.4568,-45.59,1.14
3,4,Extra Trees,Randomized tree ensemble,492.94,368.43,0.4371,-56.17,16.50
4,5,Random Forest,Tree ensemble,493.00,368.47,0.4369,-53.13,7.53
5,6,Linear Regression,Linear,616.28,460.27,0.1201,351.26,0.20
6,7,Ridge Regression,Regularized linear,616.29,460.28,0.1201,351.24,0.21
7,8,Dummy Regressor,Constant baseline,839.61,683.05,-0.6332,-522.78,0.15


Saved experiments\model_comparison_results.csv and experiments\model_comparison_results.md


## Reproducibility notes

All models use the same original `data/train.csv` through `load_data()`, the same split, and the same five features. Validation is used only to report metrics. Keep this comparison's feature and split policy fixed when interpreting results.